# 01 — EDA: разведочный анализ входных PDF-паспортов

**Задача проекта:** автономно извлекать структурированные поля из PDF-паспортов промышленного оборудования (OCR + правила + опциональная LLM).

Этот ноутбук — разведочный анализ корпуса входных документов. У нас нет классического «датасета с лейблами»: данные — это сами PDF-файлы (см. папку `project/приложения/`) и контрольная выборка с ожидаемыми значениями полей (`project/samples/control_samples.json`).

Цели EDA:
1. Понять, какие типы документов встречаются (одиночные паспорта, групповые, перечни шкафов, экранные формы).
2. Оценить, в каких документах есть embedded-текст (PDF-текстовый слой), а где придётся идти через OCR.
3. Посмотреть распределение по числу страниц, размеру файла, наличию ключевых полей.
4. Прикинуть, какие сложности встретятся при extraction (шумные сканы, нестандартные структуры, низкое разрешение).

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

DOCS_DIR = PROJECT_ROOT / 'приложения'
SAMPLES_FILE = PROJECT_ROOT / 'samples' / 'control_samples.json'

print('Корень проекта:', PROJECT_ROOT)
print('Документы:', DOCS_DIR)
print('Контрольный набор:', SAMPLES_FILE)

## 1. Корпус документов: общий обзор

In [ ]:
docs = sorted(DOCS_DIR.glob('*.pdf'))
print(f'Всего PDF: {len(docs)}\n')
for d in docs:
    print(f'  {d.name:<90} {d.stat().st_size / 1024:8.1f} KB')

## 2. Структура страниц и наличие embedded-текста

Если в PDF уже зашит текстовый слой — можно сэкономить на OCR. Если нет (скан) — нужен полноценный OCR-проход.

In [ ]:
import fitz  # PyMuPDF

rows = []
for path in docs:
    pdf = fitz.open(str(path))
    pages = pdf.page_count
    embedded_chars = sum(len((pdf[i].get_text('text') or '').strip()) for i in range(pages))
    pdf.close()
    rows.append({
        'file': path.name,
        'pages': pages,
        'embedded_chars': embedded_chars,
        'avg_chars_per_page': embedded_chars // max(1, pages),
        'mostly_scan': embedded_chars < 120 * pages,
    })

for r in rows:
    print(r)

**Что смотрим:**
- `embedded_chars` низкий → PDF почти наверняка скан → OCR обязателен.
- `embedded_chars` высокий → текстовый слой есть → можно сначала попробовать прочитать его, а OCR оставить как fallback (порог `OCR_IF_EMBEDDED_CHARS=120` в конфиге как раз про это).
- `pages` разное → пайплайн должен корректно обрабатывать многостраничные документы и не падать на однопотиях.

## 3. Контрольная выборка: распределение ожидаемых полей

In [ ]:
with SAMPLES_FILE.open('r', encoding='utf-8') as f:
    control = json.load(f)

samples = control.get('samples', [])
print(f'Всего контрольных примеров: {len(samples)}')
print(f'Целевые поля (fields): {control.get("fields", [])}\n')

field_presence = Counter()
doc_types = Counter()
for s in samples:
    exp = s.get('expected', {})
    for k, v in exp.items():
        if v:
            field_presence[k] += 1
    if 'document_type' in exp:
        doc_types[exp['document_type']] += 1

print('Заполненность полей в контроле:')
for k, v in field_presence.most_common():
    print(f'  {k:<20} {v}')

print('\nРаспределение типов документов:')
for k, v in doc_types.most_common():
    print(f'  {k:<20} {v}')

## 4. Выводы EDA

- Корпус **маленький** (≈ 7 документов): это не классический ML-сетап с тысячами примеров, а реалистичный «промышленный» кейс по IDP (Intelligent Document Processing), где разнообразие важнее объёма.
- Типы документов разнородные: одиночные паспорта, групповые паспорта, перечни шкафа, экранные формы. Под каждый тип в правилах прописаны отдельные эвристики (`_extract_structured_from_text`, `parse_cabinet_document`).
- Часть документов содержит embedded-текст, часть — сканы. Пайплайн должен уметь оба сценария: смотрим на embedded и, если его мало, идём в OCR.
- Контрольная выборка покрывает 7 полей; не все поля заполнены в каждом примере. При оценке мы сравниваем только непустые `expected` (это честный подход: метрика отражает качество там, где есть эталон).

Следующий шаг: **02_baselines.ipynb** — сравнение baseline (OCR + правила) против улучшенного варианта (OCR + правила + LLM) на этой контрольной выборке.